In [ ]:
import chromadb
from chromadb.utils import embedding_functions
import re
from pathlib import Path
from dotenv import load_dotenv
import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

## Creating VectorDB

In [ ]:
FILE_PATH = "Insert Path to Textbook"
CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "Vector_DB"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2" 

In [ ]:
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    markdown_text = f.read()

headers_to_split_on = [("##", "Chapter"), ("###", "Section")]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)
chunks_by_header = markdown_splitter.split_text(markdown_text)

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n", " ", ""])
final_chunks = recursive_splitter.split_documents(chunks_by_header) 

documents = []
metadatas = []
ids = []
source_name = os.path.basename(FILE_PATH)
id_counter = 0

for chunk in final_chunks:
    documents.append(chunk.page_content)
    chunk.metadata["source"] = source_name 
    metadatas.append(chunk.metadata)
    ids.append(f"{source_name}_{id_counter}")
    id_counter += 1
    
client = chromadb.PersistentClient(path=CHROMA_DIR)
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL_NAME)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function 
)

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

## Adding Textbooks

In [ ]:
CHROMA_DIR = "Insert Path to Textbook"
COLLECTION_NAME = "RAG_DB"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
FILE_PATH = "Insert Path to VectorDB" 
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL_NAME
)

client = chromadb.PersistentClient(path=CHROMA_DIR)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function 
)

initial_count = collection.count()


documents = []
metadatas = []
ids = []

source_name = os.path.basename(FILE_PATH)

id_counter = initial_count 

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    markdown_text = f.read()

headers_to_split_on = [
    ("##", "Chapter"), 
    ("###", "Section") 
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],
)

chunks_by_header = markdown_splitter.split_text(markdown_text)
final_chunks = recursive_splitter.split_documents(chunks_by_header) 


for chunk in final_chunks:
    documents.append(chunk.page_content)
    
    # Apply source tag
    chunk.metadata["source"] = source_name 
    metadatas.append(chunk.metadata)
    
    # Create a unique ID
    ids.append(f"{source_name}_{id_counter}")
    id_counter += 1
    
# --- 4. ADD TO COLLECTION ---

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

## Adding Articles or Youtube Videos

In [ ]:
CHROMA_DIR = "Insert Path to Textbook"
COLLECTION_NAME = "RAG_DB"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
FILE_PATH = "Insert Path to Article/Youtube Video" 
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL_NAME
)

client = chromadb.PersistentClient(path=CHROMA_DIR)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function 
)

initial_count = collection.count()

documents = []
metadatas = []
ids = []

source_name = os.path.basename(FILE_PATH)

single_id_counter = initial_count 

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    markdown_text = f.read()


documents.append(markdown_text)

metadatas.append({
    "source": source_name,
    "file_size_bytes": len(markdown_text.encode('utf-8'))
})

ids.append(f"{source_name}_full_{single_id_counter}")    

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)